In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import os
import requests
import random


/var/folders/cb/8cyby9713t3fx6gz1zq496200000gn/T/ipykernel_83860/1638351956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
load_dotenv()


True

In [4]:
llm = ChatOpenAI()


In [5]:
# Tools
search_tool = DuckDuckGoSearchRun(region="us-en")


@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Cannot divide by zero"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation: {operation}"}

        return {"result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """Get the latest stock quote for a given symbol."""
    # Get a free API key at https://www.alphavantage.co/support/#api-key
    api_key = os.getenv("ALPHAVANTAGE_API_KEY", "demo")
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey={api_key}"
    r = requests.get(url)
    return r.json()


ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.

In [ ]:
# Make tool list
tools = [get_stock_price, search_tool, calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)


In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [ ]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)  # Executes tool calls


In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)


In [ ]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")


In [ ]:
chatbot = graph.compile()
chatbot.get_graph().draw_mermaid_png()


In [ ]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)


In [ ]:
# Tool call: calculator
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is 25 multiplied by 4?")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


In [ ]:
# Tool call: web search
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is the latest news about LangGraph?")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


In [ ]:
# Tool call: stock price
out = chatbot.invoke({
    "messages": [HumanMessage(content="What is the current stock price of AAPL?")]
})

for message in out["messages"]:
    print(f"{type(message).__name__}: {message.content}")


In [ ]:
# Inspect the tool calls made by the LLM
for message in out["messages"]:
    if getattr(message, "tool_calls", None):
        print(message.tool_calls)
